# Encoder model 

<img src="encoder.png" style="filter: invert(1);" width="30%" align="right">

In this section, we will discuss encoder models, with a focus on <mark>BERT (Bidirectional Encoder Representations from Transformers)</mark> as an example. We will cover the architecture, pre-training, and fine-tuning processes.

BERT is an encoder model that uses a vocabulary of <mark>30,000 tokens</mark>. Input tokens are
converted to <mark>1024-dimensional word embeddings</mark> and passed through <mark>24 transformer
layers</mark>. Each contains a self-attention mechanism with <mark>16 heads</mark>. The queries, keys, and
values for each head are of dimension <mark>64</mark> (i.e., the matrices <mark>$\Omega_{vh}, \Omega_{qh}, \Omega_{kh}$ are 64 ×1024</mark>).

The dimension of the single hidden layer in the fully connected networks is <mark>4096</mark>. The
total number of parameters is <mark>∼340 million</mark>. When BERT was introduced, this was
considered large, but it is now much smaller than state-of-the-art models.

---


Both units are residual networks (i.e., their output is added back to the original
input). In addition, it is typical to add a LayerNorm operation after both the self-
attention and fully connected networks. This is similar to BatchNorm but normalizes
each embedding in each batch element separately using statistics calculated across its
D embedding dimensions (section 11.4 and figure 11.14). The complete layer can be
described by the following series of operations (figure 12.7):

<img src="assets/Chap12/TransformerBlock.svg" style="filter: invert(1);" width="100%">

Transformer layer. The input consists of a D × N matrix containing
the D-dimensional word embeddings for each of the N input tokens. The output is
a matrix of the same size. The transformer layer consists of a series of operations.
First, there is a multi-head attention block, allowing the word embeddings to
interact with one another. This forms the processing of a residual block, so the
inputs are added back to the output. Second, a LayerNorm operation is applied
separately to each embedding. Third, there is a second residual layer where the
same fully connected neural network is applied separately to each of the N word
representations (columns). Finally, LayerNorm is applied again.

\begin{eqnarray}
  \mathbf{X} &\leftarrow& \mathbf{X} + \mbox{\bf{MhSa}}[\mathbf{X}] \nonumber \\
  \mathbf{X} &\leftarrow& \mbox{\bf{LayerNorm}}[\mathbf{X}] \hspace{3cm}\nonumber\\
  \mathbf{x}_{n} &\leftarrow& \mathbf{x}_{n}+\mbox{\bf{mlp}}[\mathbf{x}_{n}] \hspace{3.6cm}\forall\; n\in\{1,\ldots, N\}\nonumber\\
  \mathbf{X} &\leftarrow& \mbox{\bf{LayerNorm}}[\mathbf{X}],
 \end{eqnarray}

where the column vectors xn are separately taken from the full data matrix X. In a real
network, the data passes through a series of these transformer layers.

---


## Pre-training 

<br/>

<img src="assets/Chap12/TransformerEncoder.svg" style="filter: invert(1);" width="100%">

Pre-training for BERT-like encoder. The input tokens (and a special <cls> token denoting the start of the sequence) are converted to word embeddings.

Here, these are represented as rows rather than columns, so the box labeled “word embeddings” is $\mathbf{X}^T$ . These embeddings are passed through a series of transformer layers (orange connections indicate that every token attends
to every other token in these layers) to create a set of output embeddings. 

A small fraction of the input tokens are randomly replaced with a generic <mask>
token. 

In pre-training, the goal is to predict the missing word from the associated output embedding. 

To this end, the outputs corresponding to the masked tokens are passed through softmax functions, and a multiclass classification loss is applied to each. 

This task has the advantage that it <mark>uses both the left and right context to predict the missing word</mark> but has the disadvantage that it does not make efficient use of data; here, seven tokens need to be processed to add two terms to the loss function.

In the pre-training stage, the network is trained using self-supervision. This allows the
use of enormous amounts of data without the need for manual labels. 

For BERT, the self-supervision task consists of predicting missing words from sentences from a large internet
corpus (figure 12.10). 

During training, <mark>the maximum input length is 512 tokens, and the batch size is 256.</mark> 

The system is trained for a million steps, corresponding to roughly <mark>50 epochs of the 3.3-billion word corpus.</mark>

Predicting missing words forces the transformer network to understand some syntax.

For example, it might learn that the adjective red is often found before nouns like house or car but never before a verb like shout. 

It also allows the model to learn superficial common sense about the world. For example, after training, the model will assign a
higher probability to the missing word train in the sentence The <mask> pulled into
the station than it would to the word peanut. However, the degree of “understanding”
this type of model can ever have is limited.


### Masked Language Modeling

Masked language modeling (MLM) is a pre-training objective where certain tokens in the input sequence are randomly masked, and the model is trained to predict these masked tokens based on their surrounding context.

During pre-training, the parameters of the transformer encoder are learned through **self-supervised learning** on a large corpus of text. <mark>The objective is to predict masked tokens using their surrounding context</mark>, allowing the model to learn the statistical patterns and relationships within language.

The text is first tokenized using a subword tokenization algorithm (such as BPE or WordPiece) and converted into token IDs. <mark>These token IDs are organized into fixed- or variable-length input sequences. For each sequence, **15% of token positions are randomly selected** for prediction.</mark> In the original BERT approach, 80% of these selected tokens are replaced with the special `[MASK]` token ID, 10% with a random token ID, and 10% are left unchanged.

The `[MASK]` token has its own learnable embedding vector in the embedding matrix. The modified sequence is passed through the transformer encoder, which produces a contextualized representation for every token position. At each selected position, the model uses this representation to predict the **original token ID**, which serves as the training label. The prediction error is used to update the model's parameters through backpropagation.

The goal here is for the model to learn general information about the statistics of language. 

In the fine-tuning stage, the resulting network is adapted to solve a particular task using a smaller body of labelled training data.



## Post-Training: Supervised Fine-tuning

So far, we’ve looked at **base models**, which are just pre-trained text generators. But **to make an actual assistant**, you need **post-training**.

**Base models hallucinate a lot** → They generate text, but it’s not always useful.

**Post-training fixes this** by fine-tuning the model to respond better.

The good news? **Post-training is way cheaper than pre-training** (e.g., months vs. hours).

<br/>

<img src="assets/Chap12/TransformerEncoderFineTune.svg" style="filter: invert(1);" width="100%">

After pre-training, the encoder is fine-tuned using manually labeled
data to solve a particular task. Usually, a linear transformation or a multi-layer
perceptron (MLP) is appended to the encoder to produce whatever output is
required. 

a) Example text classification task. In this sentiment classification
task, the <cls> token embedding is used to predict the probability that the
review is positive. 

b) Example word classification task. In this named entity
recognition problem, the embedding for each word is used to predict whether the
word corresponds to a person, place, or organization, or is not an entity.

In the fine-tuning stage, the model parameters are adjusted to specialize the network to
a particular task. An extra layer is appended onto the transformer network to convert
the output vectors to the desired output format. Examples include:

<mark>**Text classification**</mark>: In BERT, a special token known as the classification or `<cls>`
token is placed at the start of each string during pre-training. For text classification
tasks like <u>sentiment analysis</u> (in which the passage is labeled as having a positive or
negative emotional tone), the vector associated with the `<cls>` token is mapped to a
single number and passed through a logistic sigmoid (figure 12.11a). This contributes to
a standard binary cross-entropy loss (section 5.4).

<mark>**Word classification**</mark>: The goal of <u>named entity recognition</u> is to classify each word as an entity type (e.g., person, place, organization, or no-entity). To this end, each input
embedding $\mathbf{x}_n$ is mapped to an $E\times 1$ vector where the $E$ entries correspond to the $E$ entity types. This is passed through a softmax function to create probabilities for each class, which contribute to a multiclass cross-entropy loss (figure 12.11b).

<mark>**Text span prediction**</mark>: In the SQuAD 1.1 question answering task, the question and a
passage from Wikipedia containing the answer are concatenated and tokenized. BERT
is then used to predict the text span in the passage that contains the answer. Each
token maps to two numbers indicating how likely it is that the text span begins and
ends at this location. The resulting two sets of numbers are put through two softmax
functions. The likelihood of any text span being the answer can be derived by combining
the probability of starting and ending at the appropriate places.
